In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/processed/04_abt_seleccion.parquet")

X = df.drop(columns=["client_id", "TGT"])
y = df["TGT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("Train:", X_train.shape, "| tasa TGT=1:", round(y_train.mean(), 4))
print("Test: ", X_test.shape, "| tasa TGT=1:", round(y_test.mean(), 4))

Train: (18552, 50) | tasa TGT=1: 0.2942
Test:  (4639, 50) | tasa TGT=1: 0.2942


In [3]:
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

def calcular_auc(y_real, score):
    return roc_auc_score(y_real, score)

def calcular_ks(y_real, score):
    fpr, tpr, _ = roc_curve(y_real, score)
    return max(tpr - fpr)

def calcular_lift(y_real, score, top=0.10):
    tabla = pd.DataFrame({"y": np.asarray(y_real), "score": score})
    tabla = tabla.sort_values("score", ascending=False)
    n_top = int(len(tabla) * top)
    tasa_top = tabla["y"].head(n_top).mean()
    tasa_base = tabla["y"].mean()
    return tasa_top / tasa_base

In [4]:
score_azar = np.random.RandomState(42).rand(len(y_test))

print("AUC:     ", round(calcular_auc(y_test, score_azar), 3))
print("KS:      ", round(calcular_ks(y_test, score_azar), 3))
print("Lift 10%:", round(calcular_lift(y_test, score_azar, top=0.10), 2))
print("Lift 20%:", round(calcular_lift(y_test, score_azar, top=0.20), 2))

AUC:      0.501
KS:       0.02
Lift 10%: 1.05
Lift 20%: 1.02


In [5]:
print("Columnas con nulos:", X_train.isna().any().sum())
print("Columnas de texto:", X_train.select_dtypes(include="object").shape[1])

Columnas con nulos: 0
Columnas de texto: 0


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

modelo_log = Pipeline([
    ("escalar", StandardScaler()),
    ("logistica", LogisticRegression(max_iter=1000)),
])

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

score_cv = cross_val_predict(
    modelo_log, X_train, y_train, cv=folds, method="predict_proba"
)[:, 1]

print("AUC:     ", round(calcular_auc(y_train, score_cv), 3))
print("KS:      ", round(calcular_ks(y_train, score_cv), 3))
print("Lift 10%:", round(calcular_lift(y_train, score_cv, top=0.10), 2))
print("Lift 20%:", round(calcular_lift(y_train, score_cv, top=0.20), 2))

AUC:      0.82
KS:       0.495
Lift 10%: 2.44
Lift 20%: 2.25


In [7]:
from lightgbm import LGBMClassifier

modelo_lgbm = LGBMClassifier(random_state=42, verbose=-1)

score_cv_lgbm = cross_val_predict(
    modelo_lgbm, X_train, y_train, cv=folds, method="predict_proba"
)[:, 1]

print("AUC:     ", round(calcular_auc(y_train, score_cv_lgbm), 3))
print("KS:      ", round(calcular_ks(y_train, score_cv_lgbm), 3))
print("Lift 10%:", round(calcular_lift(y_train, score_cv_lgbm, top=0.10), 2))
print("Lift 20%:", round(calcular_lift(y_train, score_cv_lgbm, top=0.20), 2))

AUC:      0.834
KS:       0.509
Lift 10%: 2.48
Lift 20%: 2.28


## Paso 5: búsqueda de hiperparámetros (Randomized Search)

Es **evaluación para elegir**: cada combinación se mide con CV sobre train, con los mismos `folds` que la logística. El test sigue cerrado.

Se prueban 30 combinaciones al azar × 5 folds = 150 entrenamientos, y gana la de mejor AUC.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

perillas = {
    "n_estimators":      randint(100, 1000),   # cuántos árboles
    "num_leaves":        randint(15, 64),      # qué tan grande es cada árbol
    "learning_rate":     loguniform(0.01, 0.2),# qué tan rápido aprende
    "min_child_samples": randint(20, 200),     # mínimo de clientes por hoja
    "subsample":         uniform(0.6, 0.4),    # % de clientes que ve cada árbol (0,6 a 1)
    "subsample_freq":    [1],                  # necesario para que subsample se aplique
    "colsample_bytree":  uniform(0.6, 0.4),    # % de variables que ve cada árbol (0,6 a 1)
}

busqueda = RandomizedSearchCV(
    estimator=LGBMClassifier(random_state=42, verbose=-1),
    param_distributions=perillas,
    n_iter=30,
    scoring="roc_auc",
    cv=folds,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
)

busqueda.fit(X_train, y_train)

print("Mejor AUC en CV:", round(busqueda.best_score_, 4))
print("Mejores perillas:", busqueda.best_params_)

### Las 30 combinaciones, de mejor a peor

`mean_train_score` es la nota con clientes que el modelo vio; `mean_test_score`, con los que no vio. Si la primera está cerca de 1 y la segunda baja, el modelo memorizó (overfitting).

In [ ]:
resultados = pd.DataFrame(busqueda.cv_results_)
columnas = ["param_n_estimators", "param_num_leaves", "param_learning_rate",
            "param_min_child_samples", "mean_train_score", "mean_test_score", "rank_test_score"]
resultados[columnas].sort_values("rank_test_score").round(4)

### El ganador con nuestra regla (AUC, KS, lift)

Mismo `cross_val_predict` que en los pasos 3 y 4, para comparar los tres modelos igual.

In [ ]:
lgbm_ajustado = LGBMClassifier(random_state=42, verbose=-1, **busqueda.best_params_)

score_cv_ajustado = cross_val_predict(
    lgbm_ajustado, X_train, y_train, cv=folds, method="predict_proba"
)[:, 1]

comparacion = pd.DataFrame({
    "Logística":          score_cv,
    "LightGBM default":   score_cv_lgbm,
    "LightGBM ajustado":  score_cv_ajustado,
}).apply(lambda s: pd.Series({
    "AUC":      calcular_auc(y_train, s),
    "KS":       calcular_ks(y_train, s),
    "Lift 10%": calcular_lift(y_train, s, top=0.10),
    "Lift 20%": calcular_lift(y_train, s, top=0.20),
})).T.round(3)

comparacion

### ¿Hacen falta tantos árboles?

El ganador usa ~500 árboles con learning rate 0,011. Probamos menos árboles con un learning rate más alto, dejando fijas las demás perillas.

In [ ]:
from sklearn.model_selection import cross_validate

perillas_fijas = {k: v for k, v in busqueda.best_params_.items()
                  if k not in ("n_estimators", "learning_rate")}

filas = []
for lr in [0.011, 0.03, 0.05, 0.1]:
    for n in [50, 100, 200, 300, 500]:
        r = cross_validate(
            LGBMClassifier(learning_rate=lr, n_estimators=n, random_state=42, verbose=-1, **perillas_fijas),
            X_train, y_train, cv=folds, scoring="roc_auc", return_train_score=True,
        )
        filas.append({"learning_rate": lr, "arboles": n,
                      "AUC clientes vistos": r["train_score"].mean(),
                      "AUC clientes no vistos": r["test_score"].mean()})

pd.DataFrame(filas).round(4)

### Conclusiones del paso 5

1. **LightGBM le gana a la logística**, por poco pero en las cuatro métricas (AUC 0,833 vs 0,820; lift 10 % 2,55 vs 2,44).
2. **Ajustar hiperparámetros no subió el AUC** (los valores por defecto ya daban 0,834), pero dio un modelo que memoriza menos: árboles chicos (16 hojas) y mínimo 148 clientes por hoja.
3. **Más árboles no es mejor.** De 100 a 500 árboles el AUC cambia 0,001, menos que la variación entre folds (~0,003). Con learning rate alto y muchos árboles, el modelo memoriza: sube la nota con clientes vistos y baja con los no vistos.
4. **Modelo elegido: LightGBM con 100 árboles y learning rate 0,05**, el más simple con el mejor rendimiento.
5. El techo de ~0,834 en CV viene de las variables, no de las perillas.

## Paso 6: modelo final y examen con el test

Entrenamos el modelo elegido con **todo el train** y abrimos el test **una sola vez**. Después de esto no se ajusta nada más.

In [ ]:
modelo_final = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=16,
    min_child_samples=148,
    subsample=0.81,
    subsample_freq=1,
    colsample_bytree=0.76,
    random_state=42,
    verbose=-1,
)

modelo_final.fit(X_train, y_train)

score_train = modelo_final.predict_proba(X_train)[:, 1]
score_test  = modelo_final.predict_proba(X_test)[:, 1]

examen = pd.DataFrame({
    "Train": [calcular_auc(y_train, score_train), calcular_ks(y_train, score_train),
              calcular_lift(y_train, score_train, 0.10), calcular_lift(y_train, score_train, 0.20)],
    "Test":  [calcular_auc(y_test, score_test), calcular_ks(y_test, score_test),
              calcular_lift(y_test, score_test, 0.10), calcular_lift(y_test, score_test, 0.20)],
}, index=["AUC", "KS", "Lift 10%", "Lift 20%"]).round(3)

examen

### Tabla por deciles en test

Ordenamos a los clientes de test por puntaje y los partimos en 10 grupos iguales. Chequeos de la cátedra: deciles del mismo tamaño y tasa de contratación **siempre bajando**.

In [ ]:
tabla = pd.DataFrame({"y": y_test.values, "score": score_test})
tabla = tabla.sort_values("score", ascending=False).reset_index(drop=True)
tabla["decil"] = tabla.index * 10 // len(tabla) + 1

deciles = tabla.groupby("decil").agg(
    clientes=("y", "size"),
    contrataron=("y", "sum"),
    tasa=("y", "mean"),
)
deciles["lift"] = deciles["tasa"] / tabla["y"].mean()
deciles.round(3)

### Conclusión del paso 6

- **AUC en test 0,840**, en línea con el CV (0,833) y con lo que obtuvieron los demás grupos. El modelo **funciona con clientes nuevos**.
- Train (0,860) queda apenas por encima de test: diferencia chica, **sin overfitting**.
- **Lift 10 % = 2,62:** en el primer decil contrata el 77 % de los clientes, contra 29 % al azar.
- La tasa baja en cada decil (77 % → 0,2 %): el modelo **ordena bien toda la lista**.

## Paso 7: explicar el modelo con SHAP

SHAP calcula, para cada cliente, cuánto **sumó o restó** cada variable a su puntaje. Promediando sobre todos los clientes de test sale el ranking de variables que más pesan y hacia qué lado empujan.

(Si no lo tenés instalado: `pip install shap`)

In [ ]:
import shap

explicador = shap.TreeExplainer(modelo_final)
valores_shap = explicador.shap_values(X_test)
if isinstance(valores_shap, list):   # algunas versiones devuelven una lista por clase
    valores_shap = valores_shap[1]

importancia = (pd.Series(np.abs(valores_shap).mean(axis=0), index=X_test.columns)
               .sort_values(ascending=False))
importancia.head(10).round(3)

### Cómo leer el gráfico

- Cada punto es un cliente; las variables están ordenadas de más a menos importante.
- **Color:** rojo = valor alto de la variable, azul = valor bajo.
- **Posición:** a la derecha de 0 le **sube** el puntaje; a la izquierda se lo **baja**.

In [ ]:
shap.summary_plot(valores_shap, X_test, max_display=10)

### Conclusiones de SHAP

1. **El tipo de tarjeta (`Tarj_pct_target`) domina**: pesa ~5 veces más que la segunda variable. Un tercio de los clientes está en una categoría con solo 6 % de contratación, y el modelo los manda al fondo de la lista.
2. **Movimiento en la caja de ahorro suma**: más plata que entra (`Credits_Amounts`), más días de uso y más débitos → mayor propensión.
3. **Edad**: los grupos de más edad tienen más propensión.
4. **Gasto con tarjeta en los últimos 3 meses** suma.
5. **Excepción: transferencias recibidas altas restan** (`Transfer_In_Amount_mean`).
6. Las 10 primeras variables explican ~73 % del total.

**Perfil para Marketing:** cliente con el tipo de tarjeta "correcto", que usa activamente su caja de ahorro y su tarjeta, y de edad media-alta.

**Nota sobre leakage:** `Tarj_pct_target` y `Region_pct_target` se calcularon en `03_abt` con todos los clientes (incluido test). Recalculadas solo con train, el resultado en test es idéntico (AUC 0,8395): las tasas casi no cambian y mantienen el mismo orden, que es lo único que usan los árboles. Igual, en `src/` el encoding debe aprenderse solo con train.

## Paso 8: el corte para la campaña

La pregunta de Marketing: **¿hasta qué decil llamo?** Para responderla se acumulan los deciles: si llamo a los deciles 1 a N, ¿cuántos llamo, cuántos contratan y qué parte de todos los interesados encuentro?

- **acierto_acum:** de los que llamé, qué % contrata.
- **captura:** de todos los que contratarían, qué % encontré.
- **puntaje_min:** el puntaje del último cliente del decil (el corte en puntaje).

In [ ]:
acum = tabla.groupby("decil").agg(
    clientes=("y", "size"),
    contrataron=("y", "sum"),
    puntaje_min=("score", "min"),
)
acum["llamados_acum"]  = acum["clientes"].cumsum()
acum["contratan_acum"] = acum["contrataron"].cumsum()
acum["acierto_acum"]   = acum["contratan_acum"] / acum["llamados_acum"]
acum["captura"]        = acum["contratan_acum"] / tabla["y"].sum()
acum["lift_acum"]      = acum["acierto_acum"] / tabla["y"].mean()
acum.round(3)

### Ganancia según el corte

Datos de negocio: cada llamada cuesta **$10.000** y cada cliente que compra deja **$100.000**.

Punto de equilibrio: conviene llamar a un decil mientras su tasa de contratación supere 10.000 / 100.000 = **10 %**.

In [ ]:
COSTO_LLAMADA = 10_000
GANANCIA_VENTA = 100_000

acum["tasa_decil"] = acum["contrataron"] / acum["clientes"]
acum["ganancia_acum"] = acum["contratan_acum"] * GANANCIA_VENTA - acum["llamados_acum"] * COSTO_LLAMADA

acum[["llamados_acum", "contratan_acum", "tasa_decil", "ganancia_acum"]].round(3)

### Conclusión del corte

- Con costo $10.000 y ganancia $100.000, el equilibrio es 10 %: **conviene llamar a los deciles 1 a 7 (puntaje ≥ 0,08)**. El decil 7 contrata 14 % (> 10 %); el 8, 5 % (< 10 %).
- En test, ese corte da la **ganancia máxima: ~$100,8 millones**, contra ~$90,1 millones llamando a todos. El modelo ahorra 1.391 llamadas que costarían más de lo que dejan.
- Si Marketing tiene **menos capacidad** que eso, se corta antes: los primeros deciles son los que más ganancia dejan por llamada.
- Supuesto: se toma que quien contrató lo hizo por la llamada. En la realidad parte habría comprado igual; eso solo se mide con un A/B test.